In [76]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from joblib import parallel_backend
import joblib

import sys
sys.path.append('..')
from backend.data_pipeline import get_preprocessor

In [77]:
df = pd.read_csv(r'../data/raw/bank-additional-full.csv', sep = ';')
df.drop(columns = ['duration', 'pdays', 'default', 'emp.var.rate', 'nr.employed'], inplace = True)
df['y'] = np.where(df['y'] == 'yes', 1, 0)
df = df.replace('unknown', np.nan)

In [78]:
X = df.drop(columns = ['y'])
y = df['y']

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
preprocessor = get_preprocessor()

In [80]:
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

In [81]:
param_grid = {
    'smote__k_neighbors': [3],
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5, 10]
}

In [83]:
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [84]:
with parallel_backend('threading'):
    grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV] END model__max_depth=10, model__min_samples_split=10, model__n_estimators=200, smote__k_neighbors=3; total time=  12.4s
[CV] END model__max_depth=10, model__min_samples_split=10, model__n_estimators=200, smote__k_neighbors=3; total time=  12.5s
[CV] END model__max_depth=10, model__min_samples_split=10, model__n_estimators=200, smote__k_neighbors=3; total time=  12.5s
[CV] END model__max_depth=10, model__min_samples_split=10, model__n_estimators=200, smote__k_neighbors=3; total time=  12.5s
[CV] END model__max_depth=10, model__min_samples_split=10, model__n_estimators=200, smote__k_neighbors=3; total time=  12.6s


In [85]:
best_pipeline = grid_search.best_estimator_

In [86]:
grid_search.best_params_

{'model__max_depth': 10,
 'model__min_samples_split': 10,
 'model__n_estimators': 200,
 'smote__k_neighbors': 3}

In [88]:
MODEL_SAVE_PATH = '../models/RF_model_v1.joblib'
joblib.dump(best_pipeline, MODEL_SAVE_PATH)

['../models/RF_model_v1.joblib']